# PyTorch shapes: one mental model

This notebook builds **one picture** and then shows that every op you've met fits into it.

1. A tensor is **a row of numbers + a shape**.
2. Every number you pass to an op answers **one of three questions**. Which question decides what `-1` means.
3. Every op either **picks**, **regroups**, **reorders**, **combines** or **collapses**.

## Contents

- **Part 1.** A tensor is a row of numbers plus a shape
- **Part 2.** Three kinds of numbers, and what `-1` means in each
- **Part 3.** Five things an op can do
  - 3.1 Pick: `x[ ]`
  - 3.2 Regroup: same numbers, new walls (`reshape`, `view`, `flatten`, `unflatten`, `unsqueeze`, `squeeze`)
  - 3.3 Reorder: move the dims around (`transpose`, `permute`, `.T`)
  - 3.4 Collapse: run along one dim (`sum`, `argmax`, `softmax`, `keepdim`)
  - 3.5 Combine: broadcasting (`+ - * /`, `masked_fill`)
  - 3.6 Combine: matrix multiply `@`
  - 3.7 Combine: join with `cat` or `stack`
- **Part 4.** Check yourself
- **Summary**

Run the cells top to bottom; every line prints a shape so you can check yourself.

In [1]:
import torch
torch.manual_seed(123)

---
## Part 1. A tensor is a row of numbers plus a shape

`torch.arange(24).reshape(2, 3, 4)` stores the numbers `0, 1, ..., 23` **in that order**. The shape `(2, 3, 4)`
only says where to put the "walls":

```
dim 0 (size 2):   [ block 0             ] [ block 1                ]
dim 1 (size 3):     [row 0][row 1][row 2]   [row 0][row 1][row 2]
dim 2 (size 4):      0 1 2 3  4 5 6 7 ...     12 13 14 15 ...    23
```

The dims are **numbered**, like the items of a Python list, from the front **and** from the back:

```
shape:            ( 2,  3,  4 )
counted forward:    0   1   2
counted backward:  -3  -2  -1
```

So for this tensor, dim `2` and dim `-1` are the **same dim**: the last one.

In [2]:
x = torch.arange(24).reshape(2, 3, 4)
print(x)
print("shape:", x.shape, " number of dims:", x.ndim)
print("size of dim 0:", x.shape[0], " size of dim -1 (the last):", x.shape[-1])

tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])
shape: torch.Size([2, 3, 4])  number of dims: 3
size of dim 0: 2  size of dim -1 (the last): 4


---
## Part 2. Three kinds of numbers, and what `-1` means in each

Every number you pass to a tensor op is one of these three kinds. **Ask which question the number answers.**

| Kind | Question it answers | Example | `-1` means |
|---|---|---|---|
| **Element index** | *Which element along this dim?* (inside `[ ]`) | `x[0]`, `x[:, -1]` | the **last element** of that dim |
| **Dim position** | *Which dim?* | `dim=-1`, `transpose(-2, -1)`, `unsqueeze(-1)` | the **last dim** |
| **Size** | *How big?* | `reshape(3, -1)`, `view(b, -1)` | **"work it out for me"** |

The first two are really the same rule as Python lists: **negative = count from the end** (of the elements,
or of the dims). Only **sizes** are different: a size can't be negative, so PyTorch reuses `-1` there to mean
"infer this one" (at most one `-1` per call).

In [3]:
x = torch.arange(24).reshape(2, 3, 4)

# Element index: which element along a dim
print(x[-1].shape)            # last element along dim 0 -> block 1, shape (3, 4)
print(x[:, -1].shape)         # last element along dim 1 -> last row of each block, shape (2, 4)

# Dim position: which dim
print(x.sum(dim=-1).shape)    # sum along the last dim (size 4) -> (2, 3)
print(x.sum(dim=0).shape)     # sum along the first dim (size 2) -> (3, 4)

# Size: how big
print(x.reshape(6, -1).shape) # 24 numbers, 6 rows -> -1 becomes 4 -> (6, 4)

torch.Size([3, 4])
torch.Size([2, 4])
torch.Size([2, 3])
torch.Size([3, 4])
torch.Size([6, 4])


**One call can use two kinds.** `x.unflatten(-1, (2, -1))`: the first `-1` is a *dim position* ("split the
last dim"), the second is a *size* ("work out the other part"). Read every number by asking which question it answers.

In [4]:
print(x.unflatten(-1, (2, -1)).shape)   # split the last dim (4) into (2, 2) -> (2, 3, 2, 2)

torch.Size([2, 3, 2, 2])


---
## Part 3. Five things an op can do

| Verb | Ops | What it does to the numbers | Numbers you pass |
|---|---|---|---|
| **Pick** | `x[ ]` | selects some | element indices |
| **Regroup** | `reshape`, `view`, `flatten`, `unflatten`, `unsqueeze`, `squeeze` | **moves nothing**, changes the walls | sizes (`reshape`/`view`), dim positions (the others) |
| **Reorder** | `transpose`, `permute`, `.T` | **moves** numbers so the dims come in another order | dim positions |
| **Combine** | `+ - * /`, `masked_fill`, `where`, `@`, `cat`, `stack` | builds a result from two tensors | dim positions (`cat`/`stack`) |
| **Collapse** | `sum`, `mean`, `max`, `argmax`, `softmax` | runs along **one** dim, other dims stay independent | a dim position (`dim=`) |

### 3.1 Pick: `x[ ]`

Inside the brackets, slots go **left to right**, one per dim.

| In a slot | Effect on that dim |
|---|---|
| integer `0`, `-1` | picks one element; the dim **disappears** |
| slice `a:b`, `:` | picks a range; the dim **stays** |
| `None` | **adds** a new dim of size 1 |
| `...` | stands for "all the dims I'm not mentioning" |
| (nothing) | trailing dims you don't mention are kept whole |

In [5]:
x = torch.arange(24).reshape(2, 3, 4)

print(x[0].shape)          # (3, 4)        integer: dim 0 gone
print(x[0:1].shape)        # (1, 3, 4)     slice: dim 0 kept
print(x[:, -1].shape)      # (2, 4)        last row of each block; dim 2 implicitly kept
print(x[..., 0].shape)     # (2, 3)        first entry of the LAST dim: ... covers the dims before it
print(x[:, None].shape)    # (2, 1, 3, 4)  new dim after dim 0
print(x[..., None].shape)  # (2, 3, 4, 1)  new dim at the end

torch.Size([3, 4])
torch.Size([1, 3, 4])
torch.Size([2, 4])
torch.Size([2, 3])
torch.Size([2, 1, 3, 4])
torch.Size([2, 3, 4, 1])


In [6]:
# Boolean mask in a slot: pick the elements where it's True.
# The mask covers as many dims as it has, and its shape must MATCH them exactly (no broadcasting).
s = torch.zeros(2, 3, 3)
m = torch.triu(torch.ones(3, 3), diagonal=1).bool()   # (3, 3), True above the diagonal

s[:, m] = -1           # ':' for dim 0, the 2-D mask for dims 1 and 2
print(s[0])
# s[m[None]] = -1      # IndexError: mask (1, 3, 3) vs tensor (2, 3, 3); indexing never stretches a 1

tensor([[ 0., -1., -1.],
        [ 0.,  0., -1.],
        [ 0.,  0.,  0.]])


### 3.2 Regroup: same numbers, new walls

Nothing moves; only the shape changes. The product of the sizes stays the same (here 24).

| Op | Arguments | Effect |
|---|---|---|
| `reshape(sizes)` / `view(sizes)` | the **full new shape**, one size per dim; one may be `-1` | any regrouping |
| `unflatten(dim, sizes)` | a dim position + the sizes to split it into | split **one** dim |
| `flatten(start, end=-1)` | dim positions | merge dims `start..end` into one |
| `unsqueeze(dim)` | a dim position | add a size-1 dim there |
| `squeeze(dim)` | a dim position | remove a size-1 dim there |

`...` and `:` are **not allowed** in `reshape`: they belong to brackets.

In [7]:
t = torch.arange(8)
print(t.reshape(2, 4))       # same numbers in reading order, walls every 4
print(t.reshape(2, -1))      # -1 is a SIZE: 8 / 2 = 4

tensor([[0, 1, 2, 3],
        [4, 5, 6, 7]])
tensor([[0, 1, 2, 3],
        [4, 5, 6, 7]])


In [8]:
x = torch.arange(24).reshape(2, 3, 4)     # (b, tokens, d)

# Split the last dim into (2, 2), three equivalent spellings:
b, n, d = x.shape
print(x.reshape(b, n, 2, 2).shape)        # spell out every size
print(x.reshape(*x.shape[:-1], 2, -1).shape)   # *x.shape[:-1] unpacks to "2, 3"
print(x.unflatten(-1, (2, -1)).shape)     # only name the dim you change

# One token before and after: the same 4 numbers, cut into 2 groups of 2
print(x[0, 0], "->", x.unflatten(-1, (2, 2))[0, 0].tolist())

# Merge them back
print(x.unflatten(-1, (2, 2)).flatten(-2).shape)   # merge the last two dims -> (2, 3, 4)

torch.Size([2, 3, 2, 2])
torch.Size([2, 3, 2, 2])
torch.Size([2, 3, 2, 2])
tensor([0, 1, 2, 3]) -> [[0, 1], [2, 3]]
torch.Size([2, 3, 4])


In [9]:
# Add / remove a size-1 dim
v = torch.tensor([1, 2, 3])               # (3,)
print(v.unsqueeze(0).shape)               # (1, 3)  same as v[None]
print(v.unsqueeze(-1).shape)              # (3, 1)  same as v[:, None] or v[..., None]

z = torch.zeros(1, 3, 1)
print(z.squeeze(0).shape)                 # (3, 1)
print(z.squeeze(-1).shape)                # (1, 3)
print(z.squeeze().shape)                  # (3,)  no argument removes EVERY size-1 dim: always pass one

# Typical use: make one example look like a batch
print(torch.tensor([15496, 11, 314, 716]).unsqueeze(0).shape)   # (1, 4) = (batch, tokens)

torch.Size([1, 3])
torch.Size([3, 1])
torch.Size([3, 1])
torch.Size([1, 3])
torch.Size([3])
torch.Size([1, 4])


### 3.3 Reorder: move the dims around

`transpose(i, j)` swaps two dims; `permute(...)` lists the new order of all dims. Arguments are **dim positions**.
Unlike regrouping, this **moves numbers**.

In [10]:
x = torch.arange(6).reshape(2, 3)
print(x)
print(x.transpose(0, 1))   # (3, 2): rows became columns, numbers moved
print(x.reshape(3, 2))     # (3, 2): same shape, but numbers stayed in reading order -> a DIFFERENT tensor

tensor([[0, 1, 2],
        [3, 4, 5]])
tensor([[0, 3],
        [1, 4],
        [2, 5]])
tensor([[0, 1],
        [2, 3],
        [4, 5]])


In [11]:
q = torch.rand(2, 6, 4, 8)            # (b, tokens, heads, head_dim)
print(q.transpose(1, 2).shape)        # (2, 4, 6, 8)  swap dims 1 and 2
print(q.transpose(-2, -1).shape)      # (2, 6, 8, 4)  swap the last two: works for any number of dims
print(q.permute(0, 2, 1, 3).shape)    # (2, 4, 6, 8)  same as transpose(1, 2)
print(torch.rand(6, 8).T.shape)       # (8, 6)        .T: 2-D only

torch.Size([2, 4, 6, 8])
torch.Size([2, 6, 8, 4])
torch.Size([2, 4, 6, 8])
torch.Size([8, 6])


In [12]:
# After a reorder, the numbers aren't in reading order in memory any more ("not contiguous").
# view refuses; reshape copies when needed; .contiguous() makes a reading-order copy.
y = torch.rand(2, 6, 4).transpose(1, 2)
print(y.is_contiguous())                     # False
print(y.reshape(2, -1).shape)                # fine
print(y.contiguous().view(2, -1).shape)      # fine
# y.view(2, -1)                              # RuntimeError

False
torch.Size([2, 24])
torch.Size([2, 24])


### 3.4 Collapse: run along one dim

`dim=` is a **dim position**: the op runs along that dim, and **every other dim is treated as independent**.

- `sum`, `mean`, `max`, `argmax` **remove** that dim (unless `keepdim=True`, which keeps it as size 1).
- `softmax` keeps the shape; values along that dim sum to 1.

In [13]:
x = torch.tensor([[1., 3.],
                  [2., 2.],
                  [0., 4.]])                 # (3, 2)

print(x.sum(dim=0))                         # down each column -> (2,)
print(x.sum(dim=-1))                        # across each row  -> (3,)
print(x.sum(dim=-1, keepdim=True).shape)    # (3, 1)
print(x / x.sum(dim=-1, keepdim=True))      # each row now sums to 1 (keepdim makes it broadcast)
print(x.argmax(dim=-1))                     # column of the max, per row

scores = torch.rand(2, 4, 3, 3)             # (b, heads, tokens, tokens)
print(torch.softmax(scores, dim=-1).sum(dim=-1))   # all 1s: every (b, head, query) row is its own distribution

tensor([3., 9.])
tensor([4., 4., 4.])
torch.Size([3, 1])
tensor([[0.2500, 0.7500],
        [0.5000, 0.5000],
        [0.0000, 1.0000]])
tensor([1, 0, 1])
tensor([[[1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000]],

        [[1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000]]])


### 3.5 Combine: broadcasting

For `+ - * /`, comparisons, `masked_fill` and `where`, shapes may differ. Line them up **from the right**:
- missing dims on the left count as size 1;
- a size-1 dim **stretches** to match;
- otherwise sizes must be equal.

In [14]:
x = torch.ones(2, 3)
print((x + torch.tensor([10, 20, 30])).shape)    # (2, 3) + (3,)   -> (2, 3)
print((x + torch.tensor([[10], [20]])).shape)    # (2, 3) + (2, 1) -> (2, 3)
print((torch.rand(3, 1) * torch.rand(2)).shape)  # (3, 1) * (2,)   -> (3, 2)
# x + torch.tensor([10, 20])                     # (2, 3) + (2,): 3 vs 2 -> error

torch.Size([2, 3])
torch.Size([2, 3])
torch.Size([3, 2])


In [15]:
# masked_fill: "replace where the mask is True". Its mask BROADCASTS (unlike a mask inside [ ]).
scores = torch.rand(2, 4, 3, 3)                            # (b, heads, tokens, tokens)
mask = torch.triu(torch.ones(3, 3), diagonal=1).bool()     # (3, 3)
print(scores.masked_fill(mask, -torch.inf)[0, 0])          # the (3, 3) mask stretches over b and heads

# Never build -inf by multiplying: 0 * -inf = nan
print(torch.triu(torch.ones(3, 3), diagonal=1) * -torch.inf)

tensor([[0.7523,   -inf,   -inf],
        [0.2239, 0.8205,   -inf],
        [0.1976, 0.5013, 0.1121]])
tensor([[nan, -inf, -inf],
        [nan, nan, -inf],
        [nan, nan, nan]])


### 3.6 Combine: matrix multiply `@`

- The **last two dims** follow the matrix rule: `(n, k) @ (k, m) -> (n, m)`.
- **All dims before them** broadcast, as in 3.5.
- A 1-D tensor on the left acts as a row, on the right as a column.

In [16]:
print((torch.rand(6, 3) @ torch.rand(3, 4)).shape)              # (6, 4)
print((torch.rand(2, 6, 3) @ torch.rand(3, 4)).shape)           # (2, 6, 4)    the 2-D one is reused per batch
print((torch.rand(2, 4, 6, 8) @ torch.rand(2, 4, 8, 6)).shape)  # (2, 4, 6, 6) one matmul per (b, head)

q = torch.rand(2, 4, 6, 8); k = torch.rand(2, 4, 6, 8)
print((q @ k.transpose(-2, -1)).shape)                           # (2, 4, 6, 6)

torch.Size([6, 4])
torch.Size([2, 6, 4])
torch.Size([2, 4, 6, 6])
torch.Size([2, 4, 6, 6])


### 3.7 Combine: join with `cat` or `stack`

- `cat` glues along an **existing** dim, which grows.
- `stack` creates a **new** dim; all inputs must have the same shape.

In [17]:
a = torch.rand(6, 2); b = torch.rand(6, 2)
print(torch.cat([a, b], dim=-1).shape)    # (6, 4)
print(torch.cat([a, b], dim=0).shape)     # (12, 2)
print(torch.stack([a, b], dim=0).shape)   # (2, 6, 2)

torch.Size([6, 4])
torch.Size([12, 2])
torch.Size([2, 6, 2])


---
## Part 4. Check yourself

For each line, decide **which kind** every number is (element index, dim position, size), predict the shape,
then run the cell. The answers are in the asserts.

In [18]:
x = torch.rand(2, 6, 8)

assert x[-1].shape == (6, 8)                       # -1: element index (last element of dim 0)
assert x[..., -1].shape == (2, 6)                  # -1: element index (last element of the last dim)
assert x.softmax(dim=-1).shape == (2, 6, 8)        # -1: dim position
assert x.sum(dim=1).shape == (2, 8)                #  1: dim position
assert x.reshape(-1, 8).shape == (12, 8)           # -1: size
assert x.unsqueeze(-1).shape == (2, 6, 8, 1)       # -1: dim position
assert x.unflatten(-1, (2, -1)).shape == (2, 6, 2, 4)   # first -1: dim position, second: size
assert x.transpose(-2, -1).shape == (2, 8, 6)      # -2, -1: dim positions
print("all predictions checked")

all predictions checked


---
## Summary

**A tensor = numbers in reading order + a shape.** Dims are numbered `0, 1, 2, ...` from the front and
`-1, -2, ...` from the back.

**Every number you pass answers one question:**

| Question | Where | `-1` |
|---|---|---|
| Which **element** along this dim? | inside `[ ]` | last element |
| Which **dim**? | `dim=`, `transpose`, `permute`, `unsqueeze`, `squeeze`, `flatten`, `cat`, `stack`, first arg of `unflatten` | last dim |
| How **big**? | `reshape`, `view`, sizes tuple of `unflatten` | work it out |

**Every op does one of five things:** pick (`[ ]`), regroup (nothing moves), reorder (numbers move),
combine (broadcasting; `@` broadcasts all but the last two dims), collapse (along one `dim=`).

**Two traps:** a mask inside `[ ]` must match exactly, `masked_fill` broadcasts; `reshape` and `transpose`
can give the same shape but different tensors.